# AI & Python for AML
A practical notebook for Product Owners & Lead Analysts

## 0. Setup
Basic imports used throughout the notebook.

In [1]:
import pandas as pd
import numpy as np

# 1. Transaction Monitoring (TM)

**What it is:**  
Systems that detect suspicious transactions using rules or models.

**Key items:**  
- Threshold rules  
- Velocity rules  
- Behavioral rules  
- False positives  
- Explainability  


# 1.1 Transaction Monitoring - **Threshold rule example**
### Daily threshold rule monitoring

In [2]:
df = pd.DataFrame({
    "customer_id": [1,1,2,2,3],
    "amount": [5000, 6000, 2000, 9000, 15000],
    "date": ["2024-01-01"]*5
})

daily_totals = df.groupby(["customer_id", "date"])["amount"].sum().reset_index()
alerts = daily_totals[daily_totals["amount"] > 10000]
alerts

,customer_id,date,amount
0,1,2024-01-01,11000
1,2,2024-01-01,11000
2,3,2024-01-01,15000


# 2. Structuring (Smurfing)

**What it is:**  
Breaking large cash deposits into smaller ones to avoid detection.

**Key items:**  
- Sub-threshold cash deposits  
- Repeated amounts  
- Short time window  
- Multiple locations  


# 2.1 Structuring Detection - Sub-threshold cash deposits
**Detect repeated sub-threshold cash deposits**

In [3]:
df = pd.DataFrame({
    "customer_id": [1,1,1,2,2,3],
    "amount": [9900, 9800, 9700, 2000, 3000, 9500],
    "type": ["cash"]*6,
    "date": ["2024-01-01"]*6
})

cash_tx = df[(df["type"] == "cash") & (df["amount"] < 10000)]
counts = cash_tx.groupby(["customer_id", "date"]).size().reset_index(name="count")
alerts = counts[counts["count"] >= 3]
alerts

,customer_id,date,count
0,1,2024-01-01,3


# 3. KYC / CDD / EDD Risk Scoring

**Key items:**  
- High-risk country  
- Cash-intensive business  
- No banking history  
- PEP  
- Sanctions exposure  

### Example: Simple scoring model


# 3.1 KYC / CDD / EDD Risk Scoring
**Simple scoring model**

In [4]:
def kyc_risk_score(country_risk, cash_business, no_history, pep):
    score = 0
    score += 40 if country_risk else 0
    score += 30 if cash_business else 0
    score += 20 if no_history else 0
    score += 50 if pep else 0
    return score

customers = [
    {"id": 1, "country_risk": True,  "cash_business": False, "no_history": True,  "pep": False},
    {"id": 2, "country_risk": False, "cash_business": True,  "no_history": False, "pep": True},
]

for c in customers:
    c["risk_score"] = kyc_risk_score(c["country_risk"], c["cash_business"], c["no_history"], c["pep"])

customers

[{'id': 1,
  'country_risk': True,
  'cash_business': False,
  'no_history': True,
  'pep': False,
  'risk_score': 60},
 {'id': 2,
  'country_risk': False,
  'cash_business': True,
  'no_history': False,
  'pep': True,
  'risk_score': 80}]

# 4. Sanctions Screening (Name Matching)

**Key items:**  
- Fuzzy matching  
- Aliases  
- Transliteration  
- False positives  

### Example: Fuzzy name matching


# 4. Sanctions Screening (Fuzzy Matching)

In [13]:
!pip install fuzzywuzzy[speedup] python-levenshtein -q
from fuzzywuzzy import fuzz

name = "John Smith"
sanctioned = ["John Smith", "John Doe", "John Smith"]

[(s, fuzz.token_sort_ratio(name, s)) for s in sanctioned]

[('John Smith', 100), ('John Doe', 44), ('John Smith', 100)]

# 5. Anomaly Detection (Unsupervised ML)

**Key items:**  
- Isolation Forest  
- Outliers  
- False positive reduction  
- Explainability challenges  

### Example: Detect anomalies


# 5. Anomaly Detection (Isolation Forest)

In [14]:
!pip install scikit-learn -q
from sklearn.ensemble import IsolationForest

amounts = np.array([[50], [60], [55], [52], [5000], [48], [53], [49], [51]])
model = IsolationForest(contamination=0.1, random_state=42)
model.fit(amounts)

pred = model.predict(amounts)
pd.DataFrame({"amount": amounts.flatten(), "prediction": pred})

,amount,prediction
0,50,1
1,60,1
2,55,1
3,52,1
4,5000,-1
5,48,1
6,53,1
7,49,1
8,51,1


# 6. Alert Scoring / Prioritization

**Key items:**  
- Prioritization  
- Risk scoring  
- Explainability  
- SAR-driven features  

### Example: Simple scoring


# 6. Alert Scoring / Prioritization

In [15]:
alerts = pd.DataFrame({
    "alert_id": [1,2,3],
    "high_risk_country": [1,0,1],
    "sudden_spike": [1,1,0],
    "many_counterparties": [0,1,1]
})

def alert_score(row):
    return (
        50 * row["high_risk_country"] +
        30 * row["sudden_spike"] +
        20 * row["many_counterparties"]
    )

alerts["score"] = alerts.apply(alert_score, axis=1)
alerts.sort_values("score", ascending=False)

,alert_id,high_risk_country,sudden_spike,many_counterparties,score
0,1,1,1,0,80
2,3,1,0,1,70
1,2,0,1,1,50


# 7. Network / Graph Analysis

**Key items:**  
- Nodes = accounts  
- Edges = transactions  
- Detect rings and mule networks  

### Example: Connected components


# 7. Network / Graph Analysis

In [16]:
!pip install networkx -q
import networkx as nx

G = nx.Graph()
edges = [("A","B"),("B","C"),("C","D"),("A","E"),("E","F"),("X","Y")]
G.add_edges_from(edges)

list(nx.connected_components(G))

[{'A', 'B', 'C', 'D', 'E', 'F'}, {'X', 'Y'}]

# 8. NLP for AML Investigations

**Key items:**  
- Narrative classification  
- SAR drafting  
- Entity extraction  
- Adverse media  

### Example: Zero-shot classification


# 8. NLP for AML Investigations

In [17]:
!pip install transformers -q
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

text = "Customer made 12 cash deposits of $9,900 over 3 days."
labels = ["Structuring", "Money Mule", "Fraud", "Sanctions Evasion"]

classifier(text, labels)

{'sequence': 'Customer made 12 cash deposits of $9,900 over 3 days.',
 'labels': ['Money Mule', 'Structuring', 'Fraud', 'Sanctions Evasion'],
 'scores': [0.49913305044174194,
  0.24178853631019592,
  0.16794529557228088,
  0.09113315492868423]}

# 9. SAR Narrative Generation

### Example: Simple SAR narrative template


# 9. SAR Narrative Template

In [18]:
def sar_narrative(customer_id, pattern, amounts, dates):
    return (
        f"Customer {customer_id} exhibited {pattern} behavior. "
        f"Transactions totaling {sum(amounts)} occurred on {', '.join(dates)}. "
        f"Activity appears inconsistent with known customer profile."
    )

sar_narrative(1, "structuring", [9900, 9800, 9700], ["2024-01-01","2024-01-02","2024-01-03"])

'Customer 1 exhibited structuring behavior. Transactions totaling 29400 occurred on 2024-01-01, 2024-01-02, 2024-01-03. Activity appears inconsistent with known customer profile.'